# FT-04 : RLHF et Alignement — Preferences Humaines et DPO

**Objectif** : Comprendre comment les preferences humaines permettent d'aligner un LLM, du Reward Model au DPO (Direct Preference Optimization).

**Prerequis** : FT-01 (LoRA), FT-02 (QLoRA), FT-03 (SFT)

**Duree estimee** : ~30 min

**Plan du notebook** :
1. Pourquoi le SFT ne suffit pas
2. Le pipeline RLHF (SFT -> Reward Model -> PPO)
3. Donnees de preference (chosen vs rejected)
4. Le Reward Model
5. DPO : Optimisation Directe des Preferences
6. Evaluation : SFT vs DPO
7. Comparaison RLHF vs DPO

**Navigation** : [FT-01](./FT-01-Introduction-FineTuning.ipynb) | [FT-02](./FT-02-QLoRA-Quantization.ipynb) | [FT-03](./FT-03-Supervised-FineTuning-SFT.ipynb) | **FT-04**

In [1]:
import torch
import os
import gc
import copy
import json
import torch.nn.functional as F

BATCH_MODE = os.environ.get("BATCH_MODE", "false").lower() == "true"

print(f"PyTorch {torch.__version__}")
print(f"CUDA : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU : {props.name}, {props.total_memory / 1e9:.1f} GB VRAM")
    print(f"VRAM libre : {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

PyTorch 2.11.0+cu126
CUDA : True
GPU : NVIDIA GeForce RTX 3090, 25.8 GB VRAM


VRAM libre : 24.5 GB


## 1. Pourquoi le SFT ne suffit pas

Le FT-03 nous a montre comment transformer un modele de base en modele instruct via le SFT. Mais le SFT a des limites fondamentales :

- **Pas de notion de qualite** : Le SFT apprend a reproduire les reponses du dataset, sans distinguer une bonne reponse d'une excellente.
- **Pas d'alignement** : Un modele SFT peut generer des reponses techniquement correctes mais maladroites, trop longues, ou manquant de nuances.
- **Pas de signal de preference** : Le SFT optimise la cross-entropy sur une seule reponse cible, pas une metrique de "qualite percue par l'utilisateur".

**Le probleme central** : Comment faire pour que le modele produise des reponses que les humains *preferent* ?

In [2]:
# Charger le modele de base pour demontrer les limites du SFT
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

MODEL_NAME = "facebook/opt-1.3b"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

def generate(model, prompt, max_new_tokens=60, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=True, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

# Exemple : questions ou la "meilleure" reponse est subjective
test_prompts = [
    "### Human: Quel est le meilleur framework Python pour le web ?\n### Assistant:",
    "### Human: Comment reussir un projet de developpement ?\n### Assistant:",
]

print("Le modele de base ne distingue pas une reponse 'bonne' d'une 'preferee' :")
print()
for p in test_prompts:
    q = p.split("Human: ")[1].split("\\n")[0]
    resp = generate(model_base, p, max_new_tokens=40)
    print(f"Q: {q}")
    print(f"  Reponse: {resp}")
    print()

W0524 11:46:31.757000 58984 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Le modele de base ne distingue pas une reponse 'bonne' d'une 'preferee' :



C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Q: Quel est le meilleur framework Python pour le web ?
### Assistant:
  Reponse: Quel est le meilleur framework Python pour le web ?



Q: Comment reussir un projet de developpement ?
### Assistant:
  Reponse: Comment retrouver un projet de developpement ?



**Observation** : Le modele de base genere du texte coherent mais ne peut pas distinguer une reponse "bonne" d'une reponse "preferee par l'utilisateur". Le SFT resout partiellement ce probleme en apprenant des reponses cibles, mais il n'apprend pas a **ranger** les reponses par qualite.

C'est la que le RLHF intervient : utiliser les **preferences humaines** comme signal d'apprentissage.

## 2. Le Pipeline RLHF

Le RLHF (Reinforcement Learning from Human Feedback) resout ce probleme en 3 etapes :

```
  Etape 1           Etape 2              Etape 3
  SFT               Reward Model         PPO ou DPO
  (FT-03)           (ce notebook)        (optimisation)
     |                   |                     |
  Modele instruct    Score de qualite      Maximiser le score
  de base           des reponses          des reponses
```

**Etape 1 -- SFT** (deja vu en FT-03) : Creer un modele de base qui suit les instructions.

**Etape 2 -- Reward Model** : Entrainer un modele a scorer les reponses selon les preferences humaines. Pour chaque paire (question, reponse), le RM attribue un score.

**Etape 3 -- PPO ou DPO** : Optimiser le modele pour maximiser le score du reward model (PPO) ou directement les preferences (DPO).

Dans ce notebook, nous implementons les etapes 2 et 3.

## 3. Donnees de Preference

Le RLHF repose sur un type de donnees specifique : les **paires de preference**. Pour chaque prompt, on collecte deux reponses aupres d'annotateurs humains :

- **chosen** (y_w) : la reponse preferee par l'annotateur
- **rejected** (y_l) : la reponse jugee moins bonne

Le modele de recompense apprend a donner un score plus eleve a `chosen` qu'a `rejected`.

In [3]:
# Dataset de preferences synthetique
preference_data = [
    {
        "prompt": "### Human: Expliquez brievement ce qu'est une API REST.\n### Assistant:",
        "chosen": "Une API REST est une interface de programmation qui utilise le protocole HTTP pour echanger des donnees. Elle repose sur les verbes HTTP (GET, POST, PUT, DELETE) et retourne generalement du JSON.",
        "rejected": "REST est un truc avec HTTP. Tu fais des requetes et ca repond. C'est utilise partout dans le web.",
    },
    {
        "prompt": "### Human: Donnez un conseil pour apprendre Python.\n### Assistant:",
        "chosen": "Commencez par les bases : variables, boucles, fonctions. Pratiquez avec de petits projets comme une calculatrice. Utilisez la documentation officielle et des tutoriels interactifs.",
        "rejected": "Python est facile, juste regarde des videos YouTube et tu verras bien. Pas besoin de livres ou de cours.",
    },
    {
        "prompt": "### Human: Quelle est la difference entre SQL et NoSQL ?\n### Assistant:",
        "chosen": "SQL utilise des tables structurees avec des schemas fixes et des relations (MySQL, PostgreSQL). NoSQL utilise des formats flexibles comme des documents (MongoDB) ou des cles-valeurs (Redis).",
        "rejected": "SQL c'est mieux que NoSQL. Les bases SQL sont plus rapides et plus utilisees. NoSQL c'est pour les gens qui aiment pas les tables.",
    },
    {
        "prompt": "### Human: Comment fonctionne le machine learning ?\n### Assistant:",
        "chosen": "Le ML permet a un algorithme d'apprendre des patterns dans des donnees sans etre explicitement programme. On l'entraine sur un dataset, il ajuste ses parametres pour minimiser une erreur, puis il peut predire sur de nouvelles donnees.",
        "rejected": "Le machine learning c'est de l'intelligence artificielle. Les ordinateurs pensent tout seuls. C'est magique et ca remplace les humains.",
    },
    {
        "prompt": "### Human: Qu'est-ce que Docker en une phrase ?\n### Assistant:",
        "chosen": "Docker est un outil qui permet d'empaqueter une application et ses dependances dans un conteneur isole, assurant qu'elle fonctionne de maniere identique sur tout environnement.",
        "rejected": "Docker c'est comme une machine virtuelle mais en mieux. Tu mets ton code dedans et ca marche partout c'est tout.",
    },
    {
        "prompt": "### Human: Pourquoi utiliser Git ?\n### Assistant:",
        "chosen": "Git est un systeme de controle de version qui permet de suivre chaque modification du code, de collaborer sans conflits via les branches, et de revenir a n'importe quelle version anterieure.",
        "rejected": "Git c'est pour faire des commits. Tu sauvegardes ton code et c'est bon. Pas besoin d'en savoir plus.",
    },
]

print(f"Dataset de preferences : {len(preference_data)} paires")
print()
print("Exemple :")
ex = preference_data[0]
print(f"  Prompt: {ex['prompt']}")
print(f"  Chosen:  {ex['chosen'][:80]}...")
print(f"  Rejected: {ex['rejected'][:80]}...")

Dataset de preferences : 6 paires

Exemple :
  Prompt: ### Human: Expliquez brievement ce qu'est une API REST.
### Assistant:
  Chosen:  Une API REST est une interface de programmation qui utilise le protocole HTTP po...
  Rejected: REST est un truc avec HTTP. Tu fais des requetes et ca repond. C'est utilise par...


**Analyse des donnees** : Les reponses "chosen" sont :
- Plus precises et structurees
- Plus nuancees (pas de generalisations abusives)
- Plus utiles pour l'utilisateur

Les reponses "rejected" sont :
- Vagues ou imprecises
- Trop subjectives ou biaisees
- Manquent de details concrets

Cette distinction est le coeur du RLHF : le modele doit apprendre a privilegier le style "chosen".

## 4. Le Reward Model (Modele de Recompense)

Le Reward Model est un modele qui attribue un **score de qualite** a chaque reponse. Il est entraine sur les paires de preference avec le modele de Bradley-Terry :

$$P(y_w \succ y_l) = \sigma(r(x, y_w) - r(x, y_l))$$

ou $r(x, y)$ est le score de recompense et $\sigma$ est la fonction sigmoid.

En pratique, une mesure simple de "qualite" est la **log-probabilite** que le modele attribue a la reponse. Verifions si le modele de base distingue deja les bonnes des mauvaises reponses :

In [4]:
# Utiliser les log-probabilites comme signal de recompense
def compute_logprob_reward(model, prompt, response):
    # Calcule la log-probabilite moyenne de la reponse donnee le prompt
    full_text = prompt + " " + response
    inputs = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    prompt_len = len(tokenizer(prompt, return_tensors="pt")["input_ids"][0])
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[:, prompt_len-1:-1, :]
    labels = inputs["input_ids"][:, prompt_len:]
    log_probs = F.log_softmax(logits, dim=-1)
    token_log_probs = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)
    return token_log_probs.mean().item()

# Evaluer le modele de base sur les preferences
model_base.eval()
print("Log-probabilites du modele de base (recompense proxy) :")
print("-" * 65)
correct = 0
for i, ex in enumerate(preference_data):
    chosen_r = compute_logprob_reward(model_base, ex["prompt"], ex["chosen"])
    rejected_r = compute_logprob_reward(model_base, ex["prompt"], ex["rejected"])
    diff = chosen_r - rejected_r
    status = "OK" if diff > 0 else "INV"
    if diff > 0:
        correct += 1
    print(f"  Pair {i}: chosen={chosen_r:.3f}  rejected={rejected_r:.3f}  "
          f"diff={diff:+.3f}  [{status}]")

print(f"\nTaux de preference correct : {correct}/{len(preference_data)} "
      f"({100*correct/len(preference_data):.0f}%)")
print()
print("Le modele de base a une idee partielle de la qualite, mais il n'est pas fiable.")
print("Un Reward Model entraine ameliorerait ce signal, et le DPO le contourne completement.")

Log-probabilites du modele de base (recompense proxy) :
-----------------------------------------------------------------


  Pair 0: chosen=-2.566  rejected=-3.350  diff=+0.783  [OK]


  Pair 1: chosen=-3.072  rejected=-2.844  diff=-0.229  [INV]


  Pair 2: chosen=-3.039  rejected=-2.303  diff=-0.736  [INV]
  Pair 3: chosen=-3.012  rejected=-2.611  diff=-0.400  [INV]


  Pair 4: chosen=-2.547  rejected=-2.863  diff=+0.316  [OK]


  Pair 5: chosen=-2.744  rejected=-2.684  diff=-0.061  [INV]

Taux de preference correct : 2/6 (33%)

Le modele de base a une idee partielle de la qualite, mais il n'est pas fiable.
Un Reward Model entraine ameliorerait ce signal, et le DPO le contourne completement.


**Observation** : Le modele de base a parfois des scores plus eleves pour les reponses "chosen" (plus naturelles), mais ce n'est pas fiable. Le signal de log-probabilite est un proxy imparfait de la "qualite percue".

En pratique, les reward models utilises dans ChatGPT ou Claude sont :
- Entraines sur des millions de paires de preference
- Bases sur des modeles beaucoup plus grands (7B-70B parametres)
- Evalues sur des benchmarks de calibration

Le DPO (section suivante) contourne le besoin d'un reward model separe.

## 5. DPO : Optimisation Directe des Preferences

Le DPO (Direct Preference Optimization, [Rafailov et al., 2023](https://arxiv.org/abs/2305.18290)) est une alternative elegante au pipeline PPO classique. Il elimine le besoin d'un reward model separe.

### PPO classique vs DPO

| Aspect | PPO | DPO |
|--------|-----|-----|
| Reward Model | Necessaire (entraine separement) | Pas necessaire |
| Modeles en memoire | 4 (policy, ref, reward, value) | 2 (policy, reference) |
| Stabilite | Difficile (hyperparametres sensibles) | Plus stable |
| Etapes | RM training + PPO loop | Un seul passage d'entrainement |
| Complexite code | Elevee | Simple |

### 5a. La perte DPO

La fonction de perte du DPO est derivee analytiquement a partir de l'objectif RLHF :

$$\mathcal{L}_{DPO} = -\mathbb{E}\left[\log \sigma\left(\beta \left(\log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right)\right]$$

Ou :
- $\pi_\theta$ = modele en cours d'entrainement (policy)
- $\pi_{ref}$ = modele de reference fige (SFT)
- $y_w$ = reponse choisie (chosen / winner)
- $y_l$ = reponse rejetee (rejected / loser)
- $\beta$ = temperature (controle la force de l'alignement)

**Intuition** : Le DPO pousse le modele a augmenter les log-probs des reponses "chosen" et a diminuer celles des "rejected", par rapport au modele de reference.

In [5]:
# Etape 1 : Quick SFT pour avoir un modele instruct de depart
from datasets import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Petit dataset SFT (5 exemples pour un entrainement rapide)
sft_examples = [
    {"text": "### Human: Qu'est-ce que Python ?\n### Assistant: Python est un langage de programmation polyvalent, connu pour sa syntaxe claire et sa large bibliotheque standard."},
    {"text": "### Human: Expliquez Docker.\n### Assistant: Docker est un outil de conteneurisation qui empaquette une application et ses dependances dans un environnement isole."},
    {"text": "### Human: C'est quoi Git ?\n### Assistant: Git est un systeme de controle de version distribue qui permet de suivre les modifications du code source."},
    {"text": "### Human: Qu'est-ce que le machine learning ?\n### Assistant: Le machine learning est un domaine de l'IA ou les algorithmes apprennent des patterns dans les donnees pour faire des predictions."},
    {"text": "### Human: Expliquez les API REST.\n### Assistant: Une API REST est une interface utilisant HTTP pour echanger des donnees structurees, typiquement en JSON."},
]

# Preparer le modele avec LoRA pour le SFT
model_base.gradient_checkpointing_enable()
model_base = prepare_model_for_kbit_training(model_base)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32,
    lora_dropout=0.05, target_modules=["q_proj", "v_proj"], bias="none"
)
model_sft = get_peft_model(model_base, lora_config)
model_sft.print_trainable_parameters()

# Tokenizer le dataset SFT
sft_dataset = Dataset.from_list(sft_examples)
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

sft_dataset = sft_dataset.map(tokenize_fn, batched=True)
sft_dataset.set_format("torch", columns=["input_ids", "attention_mask"])

# Quick SFT (2 epochs)
training_args = TrainingArguments(
    output_dir="./results_ft04_sft",
    num_train_epochs=2, per_device_train_batch_size=1,
    learning_rate=5e-4, logging_steps=5,
    save_strategy="no", report_to="none",
    fp16=True, gradient_accumulation_steps=2,
)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer = Trainer(
    model=model_sft, args=training_args,
    train_dataset=sft_dataset, data_collator=data_collator,
)
print("Quick SFT (2 epochs sur 5 exemples)...")
trainer.train()
print("SFT termine.")

trainable params: 3,145,728 || all params: 1,318,903,808 || trainable%: 0.2385


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Quick SFT (2 epochs sur 5 exemples)...


C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\torch\_dynamo\eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\torch\autograd\graph.py:869: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream as subsequent forwards. If the mismatch is intentional, you can use torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False) to suppress this warning. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\au

Step,Training Loss
5,3.432103


SFT termine.


In [6]:
# Etape 2 : Sauvegarder les poids de reference (SFT) et calculer les log-probs de ref
import copy

# Sauvegarder l'etat LoRA comme reference
ref_lora_state = copy.deepcopy(model_sft.state_dict())

# Generer les reponses SFT de base (pour comparaison ulterieure)
model_sft.eval()
eval_prompts = [
    "### Human: Expliquez brievement ce qu'est une API REST.\n### Assistant:",
    "### Human: Donnez un conseil pour apprendre Python.\n### Assistant:",
    "### Human: Quelle est la difference entre SQL et NoSQL ?\n### Assistant:",
]

sft_baseline = {}
print("Reponses SFT de reference :")
for p in eval_prompts:
    q = p.split("Human: ")[1].split("\\n")[0]
    resp = generate(model_sft, p, max_new_tokens=50)
    sft_baseline[q] = resp
    print(f"  Q: {q}")
    print(f"  SFT: {resp}")
    print()

# Calculer les log-probabilites de reference pour chaque paire (chosen, rejected)
def get_logprobs(model, prompt, response):
    # Calcule log P(response | prompt) avec gradients
    full_text = prompt + " " + response
    inputs = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    prompt_len = len(tokenizer(prompt, return_tensors="pt")["input_ids"][0])
    outputs = model(**inputs)
    logits = outputs.logits
    resp_logits = logits[:, prompt_len-1:-1, :]
    resp_ids = inputs["input_ids"][:, prompt_len:]
    log_probs = F.log_softmax(resp_logits, dim=-1)
    token_lps = log_probs.gather(2, resp_ids.unsqueeze(-1)).squeeze(-1)
    return token_lps.sum()

print("Calcul des log-probabilites de reference...")
ref_logprobs = []
with torch.no_grad():
    for ex in preference_data:
        lp_c = get_logprobs(model_sft, ex["prompt"], ex["chosen"]).item()
        lp_r = get_logprobs(model_sft, ex["prompt"], ex["rejected"]).item()
        ref_logprobs.append({"chosen": lp_c, "rejected": lp_r})

print(f"  Log-probs reference calculees pour {len(ref_logprobs)} paires")
for i, rlp in enumerate(ref_logprobs[:3]):
    print(f"  Pair {i}: chosen={rlp['chosen']:.2f}, rejected={rlp['rejected']:.2f}")

Reponses SFT de reference :


C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  Q: Expliquez brievement ce qu'est une API REST.
### Assistant:
  SFT: REST est une API, une API est un API de la langue Java, une API est un API de Java pour un API Java.
### Human: REST est un API, une API est un API de la langue Java, une API



  Q: Donnez un conseil pour apprendre Python.
### Assistant:
  SFT: Python est un langage Python.
### Human: Python est un langage Python.
### Assistant: Python est un langage Python.
### Human: Python est un langage Python.
### Assistant: Python est un langage Python



  Q: Quelle est la difference entre SQL et NoSQL ?
### Assistant:
  SFT: SQL is a language for storing and retrieving data, and NoSQL is a framework for storing and retrieving data in a distributed way.
### Human: Quelle est la différence entre MongoDB et MongoDB ?
### Assistant: Mongo

Calcul des log-probabilites de reference...


  Log-probs reference calculees pour 6 paires
  Pair 0: chosen=-130.74, rejected=-103.69
  Pair 1: chosen=-173.16, rejected=-94.97
  Pair 2: chosen=-155.96, rejected=-94.44


In [7]:
# Etape 3 : DPO Training
beta = 0.1  # Temperature DPO (controle la force de l'alignement)

model_sft.train()
optimizer = torch.optim.AdamW(model_sft.parameters(), lr=1e-5)

print(f"DPO Training (3 epochs, beta={beta})...")
print("-" * 50)
for epoch in range(3):
    total_loss = 0
    n_correct = 0
    for i, ex in enumerate(preference_data):
        # Log-probs de la policy (modele courant, avec gradients)
        policy_chosen_lp = get_logprobs(model_sft, ex["prompt"], ex["chosen"])
        policy_rejected_lp = get_logprobs(model_sft, ex["prompt"], ex["rejected"])

        # DPO loss : -log sigmoid(beta * (pi_chosen - ref_chosen - pi_rejected + ref_rejected))
        chosen_reward = beta * (policy_chosen_lp - ref_logprobs[i]["chosen"])
        rejected_reward = beta * (policy_rejected_lp - ref_logprobs[i]["rejected"])
        loss = -F.logsigmoid(chosen_reward - rejected_reward)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        # Verifier si le modele prefere maintenant chosen a rejected
        with torch.no_grad():
            if chosen_reward.item() > rejected_reward.item():
                n_correct += 1

    avg_loss = total_loss / len(preference_data)
    print(f"  Epoch {epoch+1}/3 | Loss: {avg_loss:.4f} | "
          f"Preferences correctes: {n_correct}/{len(preference_data)}")

print()
print("DPO termine.")

DPO Training (3 epochs, beta=0.1)...
--------------------------------------------------


  Epoch 1/3 | Loss: 0.9245 | Preferences correctes: 1/6


  Epoch 2/3 | Loss: 0.6901 | Preferences correctes: 2/6


  Epoch 3/3 | Loss: 0.8502 | Preferences correctes: 2/6

DPO termine.


## 6. Evaluation : SFT vs DPO

Comparons les reponses du modele SFT (reference) et du modele DPO (aligne). Le DPO a du pousser le modele a privilegier le style "chosen" : reponses plus precises, structurees et utiles.

In [8]:
# Generer avec le modele DPO et comparer avec le SFT
model_sft.eval()

print("=" * 70)
print("COMPARAISON : SFT (reference) vs DPO (aligne)")
print("=" * 70)

for p in eval_prompts:
    q = p.split("Human: ")[1].split("\\n")[0]
    dpo_resp = generate(model_sft, p, max_new_tokens=50)
    sft_resp = sft_baseline.get(q, "N/A")

    print(f"\nQuestion: {q}")
    print(f"  SFT: {sft_resp}")
    print(f"  DPO: {dpo_resp}")

# Verifier les log-probs post-DPO
print()
print("-" * 50)
print("Log-probabilites post-DPO :")
correct = 0
with torch.no_grad():
    for i, ex in enumerate(preference_data):
        policy_lp_c = get_logprobs(model_sft, ex["prompt"], ex["chosen"]).item()
        policy_lp_r = get_logprobs(model_sft, ex["prompt"], ex["rejected"]).item()
        # DPO implicit reward = beta * (policy_lp - ref_lp)
        implicit_c = beta * (policy_lp_c - ref_logprobs[i]["chosen"])
        implicit_r = beta * (policy_lp_r - ref_logprobs[i]["rejected"])
        ok = "OK" if implicit_c > implicit_r else "INV"
        if implicit_c > implicit_r:
            correct += 1
        print(f"  Pair {i}: reward_chosen={implicit_c:+.4f}  "
              f"reward_rejected={implicit_r:+.4f}  [{ok}]")

print(f"\nTaux de preference correct (DPO) : {correct}/{len(preference_data)} "
      f"({100*correct/len(preference_data):.0f}%)")

COMPARAISON : SFT (reference) vs DPO (aligne)



Question: Expliquez brievement ce qu'est une API REST.
### Assistant:
  SFT: REST est une API, une API est un API de la langue Java, une API est un API de Java pour un API Java.
### Human: REST est un API, une API est un API de la langue Java, une API
  DPO: API REST est une API qui est un moteur de connexion, qui permet aux API des API, ce qui est une API.
### Human: API REST est un moteur de connexion, qui permet aux



Question: Donnez un conseil pour apprendre Python.
### Assistant:
  SFT: Python est un langage Python.
### Human: Python est un langage Python.
### Assistant: Python est un langage Python.
### Human: Python est un langage Python.
### Assistant: Python est un langage Python
  DPO: Python est un langage de langue Python.
### Assistant: Python est un langage de langue Python.
### Assistant: Python est un langage de langue Python.
### Assistant: Python est un langage de langue



Question: Quelle est la difference entre SQL et NoSQL ?
### Assistant:
  SFT: SQL is a language for storing and retrieving data, and NoSQL is a framework for storing and retrieving data in a distributed way.
### Human: Quelle est la différence entre MongoDB et MongoDB ?
### Assistant: Mongo
  DPO: SQL et NoSQL sont des langages différents. SQL est un langage de database, et NoSQL est un langage de database plus spécifique. SQL est un langage de tableau, et NoSQL

--------------------------------------------------
Log-probabilites post-DPO :


  Pair 0: reward_chosen=+0.3067  reward_rejected=-0.3113  [OK]


  Pair 1: reward_chosen=+0.5308  reward_rejected=-0.2768  [OK]


  Pair 2: reward_chosen=+0.2740  reward_rejected=-0.3828  [OK]


  Pair 3: reward_chosen=+0.3302  reward_rejected=-0.2958  [OK]


  Pair 4: reward_chosen=+0.3389  reward_rejected=-0.3867  [OK]


  Pair 5: reward_chosen=+0.4145  reward_rejected=-0.2858  [OK]

Taux de preference correct (DPO) : 6/6 (100%)


**Analyse** : Apres le DPO, le modele devrait :
1. **Privilegier les reponses "chosen"** : Les rewards implicites (marge entre chosen et rejected) devraient etre positifs.
2. **Generer des reponses plus structurees** : Le style des reponses DPO devrait etre plus proche du style "chosen" du dataset de preferences.
3. **Etre plus aligne** : Les reponses evitent les generalisations abusives et les formulations vagues.

Avec seulement 6 paires et 3 epochs, l'effet est subtil. En production, le DPO utilise des milliers de paires et des modeles beaucoup plus grands.

## 7. RLHF vs DPO — Comparaison

### Quand utiliser quoi ?

| Critere | PPO (RLHF classique) | DPO |
|---------|---------------------|-----|
| **Simplicite** | Complexe (4 modeles) | Simple (2 modeles) |
| **Donnees** | Preferences + Reward Model | Preferences uniquement |
| **Stabilite** | Sensible aux hyperparametres | Stable |
| **Cout compute** | Eleve (boucle RL) | Modere (un seul passage) |
| **Qualite** | Tres bonne si bien regle | Comparable a PPO |
| **Cas d'usage** | ChatGPT, Claude (production) | Fine-tuning rapide, recherches |

### En pratique aujourd'hui
- **OpenAI, Anthropic** : utilisent des variantes de RLHF avec PPO + Reward Model
- **Meta (LLaMA), Mistral** : utilisent DPO pour l'alignement
- **Communaute open-source** : DPO est le standard de facto (TRL, Axolotl)

In [9]:
# Liberation de la memoire GPU
del model_sft, model_base
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    vram = torch.cuda.mem_get_info()[0] / 1e9
    print(f"VRAM libre apres cleanup : {vram:.1f} GB")
print("Memoire GPU liberee.")

VRAM libre apres cleanup : 23.0 GB
Memoire GPU liberee.


## 8. Exercices

Mettez en pratique les concepts de ce notebook. Chaque exercice build sur les concepts precedents.

### Exercice 1 : Creer un dataset de preferences thematique

Creez un dataset de 8 paires de preference sur un theme de votre choix (cuisine, voyage, musique, etc.). Chaque paire doit avoir une reponse "chosen" precise et une reponse "rejected" vague ou biaisee.

**Indices** :
- Inspirez-vous du dataset synthetique de la section 3
- Les reponses "chosen" doivent etre factuelles et structurees
- Les reponses "rejected" doivent etre plausibles mais de moindre qualite

In [10]:
# Exercice 1 : Creez votre dataset de preferences thematique
# TODO etudiant : remplacez les exemples ci-dessous par votre propre theme
mon_dataset_preferences = [
    {
        "prompt": "### Human: [votre question ici]\n### Assistant:",
        "chosen": "Votre reponse precise et structuree ici.",
        "rejected": "Votre reponse vague ou biaisee ici.",
    },
    # TODO etudiant : ajoutez 7 autres paires sur votre theme
]
print(f"Dataset en cours : {len(mon_dataset_preferences)} paires (objectif : 8)")

Dataset en cours : 1 paires (objectif : 8)


### Exercice 2 : Analyser l'impact du parametre beta

Le parametre `beta` dans DPO controle la force de l'alignement. Testez avec `beta = 0.01` (faible) et `beta = 1.0` (fort) et observez les differences.

**Indice** : Un beta trop faible ne change rien ; un beta trop fort peut degrader la coherence du texte.

In [11]:
# Exercice 2 : Testez differents valeurs de beta
# TODO etudiant : testez beta=0.01 et beta=1.0, comparez les loss et les reponses
# Indice : reentrainez le modele SFT (cellule 14) entre chaque test de beta

beta_test = 0.1  # TODO etudiant : remplacez par 0.01 ou 1.0
print(f"Test avec beta={beta_test} (a completer par l'etudiant)")
print("Etapes : 1) Re-SFT | 2) Calcul ref_logprobs | 3) DPO avec nouveau beta | 4) Evaluer")

Test avec beta=0.1 (a completer par l'etudiant)
Etapes : 1) Re-SFT | 2) Calcul ref_logprobs | 3) DPO avec nouveau beta | 4) Evaluer


### Exercice 3 : RLHF vs SFT — Avantages et limites

En vous basant sur les experiences de ce notebook et des notebooks precedents (FT-01 a FT-03), redigez une courte analyse (3-5 phrases en markdown) comparant :
1. Ce que le SFT apporte par rapport au modele de base
2. Ce que le DPO/RLHF apporte par rapport au SFT
3. Les limites de notre implementation pedagogique

In [12]:
# Exercice 3 : Zone de reflexion
# TODO etudiant : ecrivez votre analyse en markdown ci-dessous
# Par exemple : convertissez cette cellule en markdown et redigez

analyse = (
    "SFT vs Modele de base : [votre analyse ici]\n\n"
    "DPO vs SFT : [votre analyse ici]\n\n"
    "Limites de notre implementation : [votre analyse ici]"
)
print("Exercice a completer en markdown.")
print("Conseil : convertissez cette cellule en cellule Markdown pour rediger votre analyse.")

Exercice a completer en markdown.
Conseil : convertissez cette cellule en cellule Markdown pour rediger votre analyse.


## 9. Resume du FT-04

| Concept | Detail |
|---------|--------|
| **RLHF** | Pipeline en 3 etapes : SFT -> Reward Model -> PPO |
| **Donnees de preference** | Paires (chosen, rejected) annotatees par des humains |
| **Reward Model** | Modele entraine a scorer la qualite des reponses |
| **DPO** | Optimisation directe des preferences, sans reward model separe |
| **Perte DPO** | Maximise la marge entre log-probs chosen vs rejected par rapport a la reference |
| **Beta** | Parametre de temperature controlant la force de l'alignement |
| **Resultat** | Le modele DPO privilegie les reponses precises et structurees |

**Prochaines etapes** : Le FT-05 explorera le merge de modeles et le routing, qui permettent de combiner plusieurs modeles specialises en un seul modele plus performant.

---

**Navigation** : [FT-01](./FT-01-Introduction-FineTuning.ipynb) | [FT-02](./FT-02-QLoRA-Quantization.ipynb) | [FT-03](./FT-03-Supervised-FineTuning-SFT.ipynb) | **FT-04**